# AutoGen: 角色模型和Agent框架

AutoGen v0.4 围绕角色模型重新设计了Agent编排。让消息传递异步、事件驱动Agents、错误隔离、自然获得并发性。

## 问题描述

大多数agent框架是同步的：一个agent生产、一个agent消费，在同一个调用堆栈中。出现失败堆栈就崩溃了。并发是硬加上去的，要分布式就得重写。

AutoGen v0.4的答案：角色模型。每个agent是工作在私有域的角色，消息是唯一交流的方式。运行时将消息发送和处理解耦，错误就被隔离在一个角色中。并发是自然的，分布式只是换个传输。

## 基本概念

### 角色

一个角色由
- 一个私有状态（外部永远不能直接访问）
- 一个输入框（消息队列）
- 一个处理器，消费消息，产生行为（比如回复、转发、新建一个角色、更新状态、停止自己）

两个角色之间不能够共享内存，只能够传递消息。

### 解耦为什么重要

三个原因
1. 错误隔离。  Agent B崩溃不会连带导致Agent A崩溃，B的处理器捕获运行时崩溃然后决定下一步该干什么（日志、重试、还是通知）
2. 自然并发。  同时有许多消息在途，角色并发处理各自的收件箱。
3. 分布式就绪。  不管角色是在进程内还是另一个主机，收件箱加传输都是同一个抽象设计。

# 开始编码

对应本章核心：**角色 = 私有状态 + 收件箱 + 处理器**、**只通过消息交流**、**错误隔离**、**自然并发**。  
先用异步玩具运行时跑通多角色消息与故障隔离；再用 **LangChain + DeepSeek** 把角色挂成真实 LLM worker（不硬凑 PyTorch）。


## 1. 教学玩具：角色运行时

- **Actor**：`_state` 私有；外界只能 `tell` / `ask`。
- **Mailbox**：asyncio 队列；运行时解耦「发送」与「处理」。
- **Isolation**：处理器异常变成 `ErrorReport`，不拖垮其它角色。
- **Concurrency**：多角色各自 drain 收件箱，消息可同时在途。


In [ ]:
from __future__ import annotations

import asyncio
import threading
import time
import uuid
from concurrent.futures import Future
from dataclasses import dataclass, field
from typing import Any, Awaitable, Callable


@dataclass(frozen=True)
class Envelope:
    """角色间唯一交流载体。"""

    sender: str
    recipient: str
    kind: str
    payload: dict[str, Any]
    reply_to: str | None = None
    msg_id: str = field(default_factory=lambda: uuid.uuid4().hex[:10])


@dataclass
class ActorBehavior:
    """一次处理的副作用（只能经运行时发出）。"""

    reply: Envelope | None = None
    forward: list[Envelope] = field(default_factory=list)
    state_update: dict[str, Any] = field(default_factory=dict)
    stop: bool = False


Handler = Callable[[dict[str, Any], Envelope], Awaitable[ActorBehavior] | ActorBehavior]


class Actor:
    """角色：私有状态 + 收件箱 + 处理器。"""

    def __init__(self, name: str, handler: Handler, *, initial: dict[str, Any] | None = None) -> None:
        self.name = name
        self._state: dict[str, Any] = dict(initial or {})
        self._mailbox: asyncio.Queue[Envelope] = asyncio.Queue()
        self._handler = handler
        self._alive = True
        self.errors: list[str] = []

    @property
    def alive(self) -> bool:
        return self._alive

    def snapshot(self) -> dict[str, Any]:
        """
        Returns:
            view: 状态只读副本（演示用；生产中外部不可直接读私有域）。
        """
        return dict(self._state)

    async def deliver(self, env: Envelope) -> None:
        """投递到收件箱（不直接触碰状态）。"""
        if not self._alive:
            return
        await self._mailbox.put(env)

    async def _handle_one(self, env: Envelope) -> ActorBehavior:
        try:
            out = self._handler(self._state, env)
            if asyncio.iscoroutine(out):
                out = await out
            assert isinstance(out, ActorBehavior)
            if out.state_update:
                self._state.update(out.state_update)
            if out.stop:
                self._alive = False
            return out
        except Exception as e:  # 错误隔离：不向外抛
            self.errors.append(f"{type(e).__name__}: {e}")
            return ActorBehavior(
                reply=Envelope(
                    sender=self.name,
                    recipient=env.sender,
                    kind="ErrorReport",
                    payload={"error": str(e), "failed_kind": env.kind},
                    reply_to=env.msg_id,
                )
            )


class ActorRuntime:
    """进程内传输：发送与处理解耦；可换成跨主机传输而不改角色代码。"""

    def __init__(self) -> None:
        self.actors: dict[str, Actor] = {}
        self.trace: list[dict[str, Any]] = []
        self._tasks: list[asyncio.Task[None]] = []
        self._running = False

    def spawn(self, actor: Actor) -> None:
        """
        Args:
            actor: 新角色。
        """
        if actor.name in self.actors:
            raise ValueError(f"actor exists: {actor.name}")
        self.actors[actor.name] = actor
        if self._running:
            self._tasks.append(asyncio.create_task(self._pump(actor)))

    async def tell(self, env: Envelope) -> None:
        """
        Args:
            env: 消息信封。
        """
        self.trace.append(
            {"t": time.time(), "event": "send", "from": env.sender, "to": env.recipient, "kind": env.kind}
        )
        dst = self.actors.get(env.recipient)
        if dst is None:
            self.trace.append({"event": "drop", "to": env.recipient, "kind": env.kind})
            return
        await dst.deliver(env)

    async def _pump(self, actor: Actor) -> None:
        while self._running and actor.alive:
            try:
                env = await asyncio.wait_for(actor._mailbox.get(), timeout=0.05)
            except asyncio.TimeoutError:
                continue
            behavior = await actor._handle_one(env)
            self.trace.append(
                {
                    "event": "handle",
                    "actor": actor.name,
                    "kind": env.kind,
                    "errors": list(actor.errors[-1:]) if actor.errors else [],
                }
            )
            if behavior.reply is not None:
                await self.tell(behavior.reply)
            for fwd in behavior.forward:
                await self.tell(fwd)

    async def start(self) -> None:
        """启动所有角色的收件箱泵（自然并发）。"""
        self._running = True
        self._tasks = [asyncio.create_task(self._pump(a)) for a in self.actors.values()]

    async def stop(self) -> None:
        self._running = False
        if self._tasks:
            await asyncio.gather(*self._tasks, return_exceptions=True)
        self._tasks.clear()

    async def ask(self, env: Envelope, *, timeout: float = 2.0) -> Envelope | None:
        """
        请求-响应：挂临时 waiter 角色收回复。

        Args:
            env: 请求信封。
            timeout: 秒。

        Returns:
            reply: 响应信封或超时 None。
        """
        waiter = f"__wait_{uuid.uuid4().hex[:8]}"
        box: asyncio.Queue[Envelope] = asyncio.Queue()

        async def _wait_handler(state: dict[str, Any], msg: Envelope) -> ActorBehavior:
            await box.put(msg)
            return ActorBehavior(stop=True)

        self.spawn(Actor(waiter, _wait_handler))
        await self.tell(
            Envelope(
                sender=waiter,
                recipient=env.recipient,
                kind=env.kind,
                payload=env.payload,
                reply_to=waiter,
            )
        )
        try:
            return await asyncio.wait_for(box.get(), timeout=timeout)
        except asyncio.TimeoutError:
            return None


class BackgroundLoop:
    """常驻事件循环：同步工具与异步角色运行时之间的桥。"""

    def __init__(self) -> None:
        self.loop = asyncio.new_event_loop()
        self._thread = threading.Thread(target=self._run, name="actor-loop", daemon=True)
        self._thread.start()

    def _run(self) -> None:
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()

    def submit(self, coro: Any, *, timeout: float | None = 120.0) -> Any:
        """
        Args:
            coro: 协程。
            timeout: 等待秒数。

        Returns:
            result: 协程返回值。
        """
        fut: Future[Any] = asyncio.run_coroutine_threadsafe(coro, self.loop)
        return fut.result(timeout=timeout)


def make_assistant(name: str, *, fail_on: str | None = None) -> Actor:
    """
    Args:
        name: 角色名。
        fail_on: 若收到该 kind 则抛错（测隔离）。
    """

    async def handle(state: dict[str, Any], env: Envelope) -> ActorBehavior:
        if fail_on and env.kind == fail_on:
            raise RuntimeError(f"{name} crashed on {env.kind}")
        n = int(state.get("n", 0)) + 1
        text = f"{name}:ack#{n}:{env.payload.get('text', '')}"
        reply_to = env.reply_to or env.sender
        return ActorBehavior(
            state_update={"n": n, "last": text},
            reply=Envelope(
                sender=name,
                recipient=reply_to,
                kind="AssistantReply",
                payload={"text": text},
                reply_to=env.msg_id,
            ),
        )

    return Actor(name, handle, initial={"n": 0})


def make_router(name: str, routes: dict[str, str]) -> Actor:
    """按 payload.topic 转发；自身不共享下游状态。"""

    async def handle(state: dict[str, Any], env: Envelope) -> ActorBehavior:
        topic = str(env.payload.get("topic", "default"))
        dst = routes.get(topic, routes.get("default", ""))
        if not dst:
            return ActorBehavior()
        fwd = Envelope(
            sender=name,
            recipient=dst,
            kind="RoutedTask",
            payload=dict(env.payload),
            reply_to=env.reply_to or env.sender,
        )
        return ActorBehavior(
            state_update={"routed": int(state.get("routed", 0)) + 1},
            forward=[fwd],
        )

    return Actor(name, handle, initial={"routed": 0})


print("ActorRuntime ready | private state + mailbox + isolation")


## 2. 玩具示例：隔离、并发、只传消息


In [ ]:
async def demo_actor_runtime() -> None:
    """断言错误隔离、并发投递、无私有状态外泄。"""
    rt = ActorRuntime()
    rt.spawn(make_assistant("worker_a"))
    rt.spawn(make_assistant("worker_b", fail_on="Bomb"))
    rt.spawn(make_router("router", {"math": "worker_a", "code": "worker_b", "default": "worker_a"}))
    await rt.start()

    rep = await rt.ask(Envelope(sender="ext", recipient="worker_a", kind="Task", payload={"text": "hi"}))
    assert rep is not None and rep.kind == "AssistantReply"
    assert "worker_a:ack#1:hi" in rep.payload["text"]
    print("ask:", rep.payload["text"])

    err = await rt.ask(
        Envelope(sender="ext", recipient="worker_b", kind="Bomb", payload={}),
        timeout=1.0,
    )
    assert err is not None and err.kind == "ErrorReport"
    assert rt.actors["worker_b"].alive and rt.actors["worker_a"].alive
    rep2 = await rt.ask(Envelope(sender="ext", recipient="worker_a", kind="Task", payload={"text": "still"}))
    assert rep2 is not None and "ack#2" in rep2.payload["text"]
    print("isolation: B error=", err.payload["error"], "; A still", rep2.payload["text"])

    t0 = time.time()
    await asyncio.gather(
        rt.tell(Envelope("ext", "worker_a", "Task", {"text": "p1"})),
        rt.tell(Envelope("ext", "worker_b", "Task", {"text": "p2"})),
    )
    await asyncio.sleep(0.15)
    assert rt.actors["worker_a"].snapshot()["n"] >= 2
    assert rt.actors["worker_b"].snapshot()["n"] >= 1
    print("concurrency ok in", round(time.time() - t0, 3), "s")

    await rt.tell(Envelope("ext", "router", "UserMsg", {"topic": "math", "text": "2+2"}))
    await asyncio.sleep(0.1)
    assert rt.actors["router"].snapshot()["routed"] >= 1
    sa = rt.actors["worker_a"].snapshot()
    sa["n"] = -1
    assert rt.actors["worker_a"].snapshot()["n"] != -1
    print("no shared memory: mutating snapshot does not touch actor")

    await rt.stop()
    print("TOY DEMO OK")


def run_demo_actor_runtime() -> None:
    """notebook / 脚本均可。"""
    try:
        asyncio.get_running_loop()
        in_loop = True
    except RuntimeError:
        in_loop = False
    if in_loop:
        bg = BackgroundLoop()
        bg.submit(demo_actor_runtime())
    else:
        asyncio.run(demo_actor_runtime())


run_demo_actor_runtime()


## 3. 生产级：角色运行时 + LangChain / DeepSeek

两个 LLM 角色（researcher / writer）只经消息协作；工具暴露 `spawn_team` / `tell_actor` / `ask_actor` / `runtime_trace`。  
崩溃隔离由角色运行时保证；LLM 调用失败会变成 `ErrorReport`。需 ``DEEPSEEK_API_KEY``。


In [ ]:
import json
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
LOOP = BackgroundLoop()
PROD_RT = ActorRuntime()


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def make_llm_actor(name: str, system: str) -> Actor:
    """
    Args:
        name: 角色名。
        system: 角色系统提示。

    Returns:
        actor: 处理器内调用 DeepSeek；异常 → ErrorReport。
    """

    async def handle(state: dict[str, Any], env: Envelope) -> ActorBehavior:
        if env.kind == "Stop":
            return ActorBehavior(stop=True)
        user = env.payload.get("text") or json.dumps(env.payload, ensure_ascii=False)
        prompt = (
            f"{system}\n\nIncoming({env.kind}) from {env.sender}:\n{user}\n\n"
            "Reply in Chinese, <=80 chars."
        )
        # LLM 在线程池调，避免阻塞角色 loop
        text = await asyncio.to_thread(lambda: str(get_llm().invoke(prompt).content).strip())
        n = int(state.get("n", 0)) + 1
        reply_to = env.reply_to or env.sender
        return ActorBehavior(
            state_update={"n": n, "last": text},
            reply=Envelope(
                sender=name,
                recipient=reply_to,
                kind="AssistantReply",
                payload={"text": text, "role": name},
                reply_to=env.msg_id,
            ),
        )

    return Actor(name, handle, initial={"n": 0, "role": name})


def make_orchestrator(name: str = "orchestrator") -> Actor:
    """编排：Task → researcher → writer → FinalAnswer；ErrorReport 降级返回。"""

    async def handle(state: dict[str, Any], env: Envelope) -> ActorBehavior:
        if env.kind in ("Task", "UserMsg"):
            user = env.reply_to or env.sender
            task = env.payload.get("text", "")
            return ActorBehavior(
                state_update={"user": user, "task": task, "phase": "research"},
                forward=[
                    Envelope(
                        sender=name,
                        recipient="researcher",
                        kind="Research",
                        payload={"text": task},
                        reply_to=name,
                    )
                ],
            )
        if env.kind == "AssistantReply" and env.sender == "researcher":
            notes = env.payload.get("text", "")
            return ActorBehavior(
                state_update={"notes": notes, "phase": "write"},
                forward=[
                    Envelope(
                        sender=name,
                        recipient="writer",
                        kind="Write",
                        payload={"text": f"Task: {state.get('task')}\nNotes: {notes}"},
                        reply_to=name,
                    )
                ],
            )
        if env.kind == "AssistantReply" and env.sender == "writer":
            user = state.get("user", "user")
            return ActorBehavior(
                state_update={"phase": "done", "final": env.payload.get("text", "")},
                reply=Envelope(
                    sender=name,
                    recipient=user,
                    kind="FinalAnswer",
                    payload={"text": env.payload.get("text", ""), "notes": state.get("notes", "")},
                ),
            )
        if env.kind == "ErrorReport":
            user = state.get("user", env.sender)
            return ActorBehavior(
                reply=Envelope(
                    sender=name,
                    recipient=user,
                    kind="FinalAnswer",
                    payload={"text": f"[degraded] {env.payload.get('error')}", "degraded": True},
                )
            )
        return ActorBehavior()

    return Actor(name, handle, initial={"phase": "idle"})


async def reset_prod_runtime() -> None:
    """重建生产角色团队。"""
    global PROD_RT
    if PROD_RT._running:
        await PROD_RT.stop()
    PROD_RT = ActorRuntime()
    PROD_RT.spawn(make_orchestrator("orchestrator"))
    PROD_RT.spawn(make_llm_actor("researcher", "You are a researcher. Extract 2 short bullet facts."))
    PROD_RT.spawn(make_llm_actor("writer", "You are a writer. Turn notes into one tight Chinese sentence."))
    PROD_RT.spawn(make_assistant("fragile", fail_on="Bomb"))
    await PROD_RT.start()


class SpawnArgs(BaseModel):
    pass


class TellArgs(BaseModel):
    recipient: str
    kind: str = "Task"
    text: str = ""


class AskArgs(BaseModel):
    recipient: str
    kind: str = "Task"
    text: str = ""
    timeout: float = 60.0


class TraceArgs(BaseModel):
    last_n: int = Field(default=12, ge=1, le=50)


def spawn_team_sync() -> str:
    """同步封装：启动 orchestrator + researcher + writer + fragile。"""

    async def _go() -> dict[str, Any]:
        await reset_prod_runtime()
        return {"actors": sorted(PROD_RT.actors), "status": "started"}

    return json.dumps(LOOP.submit(_go()), ensure_ascii=False)


def tell_actor_sync(recipient: str, kind: str, text: str) -> str:
    """
    Returns:
        json: 已投递。
    """

    async def _go() -> dict[str, Any]:
        if not PROD_RT._running:
            await reset_prod_runtime()
        await PROD_RT.tell(Envelope("tool", recipient, kind, {"text": text}))
        await asyncio.sleep(0.05)
        return {"told": recipient, "kind": kind}

    return json.dumps(LOOP.submit(_go()), ensure_ascii=False)


def ask_actor_sync(recipient: str, kind: str, text: str, timeout: float = 60.0) -> str:
    """
    Returns:
        json: 响应信封。
    """

    async def _go() -> dict[str, Any]:
        if not PROD_RT._running:
            await reset_prod_runtime()
        rep = await PROD_RT.ask(
            Envelope("tool", recipient, kind, {"text": text}),
            timeout=timeout,
        )
        if rep is None:
            return {"status": "timeout"}
        return {"kind": rep.kind, "sender": rep.sender, "payload": rep.payload}

    return json.dumps(LOOP.submit(_go(), timeout=timeout + 30.0), ensure_ascii=False)


def runtime_trace_sync(last_n: int = 12) -> str:
    """
    Returns:
        json: 最近消息轨迹 + 各角色错误。
    """
    errors = {n: a.errors for n, a in PROD_RT.actors.items() if a.errors}
    snaps = {n: a.snapshot() for n, a in PROD_RT.actors.items() if not n.startswith("__wait")}
    return json.dumps(
        {"trace": PROD_RT.trace[-last_n:], "errors": errors, "snapshots": snaps},
        ensure_ascii=False,
        default=str,
    )


def build_autogen_tools() -> list[StructuredTool]:
    """
    Returns:
        tools: 角色运行时四件套。
    """

    def _spawn(**kwargs: Any) -> str:
        return spawn_team_sync()

    def _tell(**kwargs: Any) -> str:
        a = TellArgs(**kwargs)
        return tell_actor_sync(a.recipient, a.kind, a.text)

    def _ask(**kwargs: Any) -> str:
        a = AskArgs(**kwargs)
        return ask_actor_sync(a.recipient, a.kind, a.text, a.timeout)

    def _trace(**kwargs: Any) -> str:
        return runtime_trace_sync(TraceArgs(**kwargs).last_n)

    return [
        StructuredTool.from_function(
            name="spawn_team",
            description="Start actor runtime: orchestrator, researcher, writer, fragile.",
            func=_spawn,
            args_schema=SpawnArgs,
        ),
        StructuredTool.from_function(
            name="tell_actor",
            description="Fire-and-forget message to an actor mailbox.",
            func=_tell,
            args_schema=TellArgs,
        ),
        StructuredTool.from_function(
            name="ask_actor",
            description="Request-response via actor mailboxes (orchestrator for multi-agent jobs).",
            func=_ask,
            args_schema=AskArgs,
        ),
        StructuredTool.from_function(
            name="runtime_trace",
            description="Inspect recent message trace, per-actor errors, and state snapshots.",
            func=_trace,
            args_schema=TraceArgs,
        ),
    ]


AUTOGEN_TOOLS = build_autogen_tools()


def build_control_agent() -> Any:
    """
    Returns:
        agent: 通过工具驱动角色运行时。
    """
    system = (
        "You operate an AutoGen-style actor runtime via tools.\n"
        "Flow: spawn_team -> ask_actor(recipient=orchestrator, kind=Task, text=...) -> "
        "runtime_trace. For isolation demo: ask_actor(fragile, kind=Bomb) then continue.\n"
        "Reply in Chinese when summarizing."
    )
    return create_agent(get_llm(), AUTOGEN_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 500 else str(m.content)[:500] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


def count_tool_calls(messages: list[BaseMessage]) -> int:
    n = 0
    for m in messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            n += len(m.tool_calls)
    return n


print(f"AutoGen-style actors + LangChain ready | {MODEL}")


## 4. 生产示例：多角色消息协作 + 故障隔离


In [ ]:
def demo_deepseek_autogen() -> None:
    """真实 API；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production demo: DEEPSEEK_API_KEY missing")
        return

    print("=== spawn + isolation ===")
    print(spawn_team_sync())
    bomb = json.loads(ask_actor_sync("fragile", "Bomb", "x", timeout=5.0))
    assert bomb.get("kind") == "ErrorReport"
    print("fragile ErrorReport:", bomb)

    print("=== orchestrator pipeline ===")
    final = json.loads(
        ask_actor_sync(
            "orchestrator",
            "Task",
            "用一句话解释：角色模型为何能隔离 agent 错误？",
            timeout=90.0,
        )
    )
    print(json.dumps(final, ensure_ascii=False, indent=2)[:800])
    assert final.get("kind") == "FinalAnswer"
    assert final.get("payload", {}).get("text")
    # degraded=True 也算通过：证明 ErrorReport 被编排角色吞掉而非拖垮运行时

    print("=== control agent ===")
    try:
        agent = build_control_agent()
        result = agent.invoke(
            {
                "messages": [
                    HumanMessage(
                        content=(
                            "spawn_team，然后 ask_actor 给 orchestrator 一个短任务："
                            "解释消息传递为何利于并发。最后 runtime_trace 并用中文总结。"
                        )
                    )
                ]
            }
        )
        print(format_agent_messages(result["messages"]))
        assert count_tool_calls(result["messages"]) >= 2
    except Exception as e:
        # 控制面 LLM 连不上时，前面的角色管道已覆盖核心要点
        print(f"control agent skipped due to LLM error: {type(e).__name__}: {e}")
    print("PRODUCTION DEMO OK")


demo_deepseek_autogen()
